# 06_labeling_and_event_generation.ipynb

This notebook implements the target labeling and event generation for the machine learning models.

### Objectives:
1. Load the engineered feature store.
2. Filter and define trade events (e.g. liquidity sweep + FVG alignment).
3. Apply the **Triple-Barrier Method** to generate labels (+1, 0, -1) with dynamic ATR barriers.
4. Generate **Meta-Labels** (1 for success, 0 for failure) for the second-stage model.
5. Save the events and labels to Google Drive and analyze the label distribution.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import sys
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Load Configurations & Features

In [ ]:
from utils.config import load_global_config
from utils.io_utils import load_parquet

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")

features_path = os.path.join(PROJECT_ROOT, "data", "feature_store", f"{symbol}_5m_features.parquet")
df = load_parquet(features_path)
print(f"Loaded feature dataset with shape: {df.shape}")

## 3. Define Events (Setup Triggers)

Instead of labeling every single bar, we label only when a potential trading setup is triggered (Event-Based Labeling).
We define a bullish event as: a bullish liquidity sweep OR a high bullish confluence score.
We define a bearish event as: a bearish liquidity sweep OR a high bearish confluence score.

In [ ]:
import numpy as np

# Bullish triggers
bullish_events = df.index[df["bullish_sweep_24h"] | (df["confluence_bullish_score"] >= 4.0)]
# Bearish triggers
bearish_events = df.index[df["bearish_sweep_24h"] | (df["confluence_bearish_score"] >= 4.0)]

all_events = bullish_events.union(bearish_events)
print(f"Generated {len(bullish_events)} bullish events, {len(bearish_events)} bearish events. Total: {len(all_events)} events.")

## 4. Apply Triple-Barrier Labeling

In [ ]:
from labeling.triple_barrier import apply_triple_barrier
from labeling.meta_labeling import generate_meta_labels

tp_mult = config.get("labeling", "triple_barrier", {}).get("tp_mult", 2.0)
sl_mult = config.get("labeling", "triple_barrier", {}).get("sl_mult", 1.0)
time_barrier = config.get("labeling", "triple_barrier", {}).get("time_barrier", 24)

df_barriers = apply_triple_barrier(
    df=df,
    events=all_events,
    tp_mult=tp_mult,
    sl_mult=sl_mult,
    time_barrier=time_barrier
)

# Add meta-labels
df_barriers["meta_label"] = generate_meta_labels(df_barriers)

print("Label distribution:")
print(df_barriers["label"].value_counts())
print("\nMeta-label distribution:")
print(df_barriers["meta_label"].value_counts())

## 5. Save Events and Labels

In [ ]:
from utils.io_utils import save_parquet

labels_path = os.path.join(PROJECT_ROOT, "data", "labels", f"{symbol}_5m_labels.parquet")
save_parquet(df_barriers, labels_path)
print(f"Saved labels to {labels_path}")

## Summary of Generated Labels

We generated:
1. `data/labels/BTCUSDT_5m_labels.parquet` containing the triple-barrier labels (+1, 0, -1) and meta-labels (1, 0).
2. Analysis of the label balance to ensure we have sufficient instances of each class for training.

**Next Step**: Run [07_model_training_baselines_and_ml.ipynb](file:///content/drive/MyDrive/btcusdt_quant_research/notebooks/07_model_training_baselines_and_ml.ipynb) to train machine learning models on these labeled events.